# **Prediction of Solar Energy Potential Based on Weather and Location Data**

## 📌 Problem Statement

The US Department of Energy projects electricity demand to grow by roughly **30% by 2035**.
Meeting that growth with renewables is the central infrastructure question of the decade, and
solar is the fastest lever: the same report projects solar rising from about **3% of US
electricity supply today to as much as 40% by 2035**.

Solar is intermittent, and its output at a given moment depends on weather, time of day, season,
and site elevation. Building capacity where the resource is weak wastes capital; building it
where the resource is strong and predictable pays back. The bottleneck is therefore not panel
technology but **site and condition-level yield forecasting**.

## 📌 Goals

- **Main** — predict the power output (`PolyPwr`, Watts) of a horizontal photovoltaic panel from
  weather, location, and time-of-observation features.
- **Secondary** — rank sites and conditions by expected output so that new capacity can be
  directed toward the highest-yield locations.

## 📌 Objectives

- Quantify which weather and temporal features actually drive panel output, and which are
  redundant.
- Compare a broad set of regression algorithms under one evaluation harness.
- Tune the strongest candidates and persist them for deployment.
- Leave an honest record of what the model can and cannot generalise to.

## 📌 Business & Technical Metrics

- **MAE** and **RMSE** in Watts — the error a site planner actually experiences.
- **MAPE** — relative error, reported with the caveat that it explodes as `PolyPwr` approaches zero.
- **R² (cross-validated)** — the main technical metric, and the one the source paper reports.
- **Site-level mean absolute error** — the business proxy: how far off is the expected yield we
  would quote for a site?

# **STAGE 1 : EXPLORATORY DATA ANALYSIS (EDA)**

## Import Libraries and Settings

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("darkgrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
np.random.seed(42)

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("seaborn:", sns.__version__)

### Load Dataset

In [ ]:
# Load dataset
df_raw = pd.read_csv("data/Pasion et al dataset.csv")

# Work on a copy so the raw frame stays available for reference
df = df_raw.copy()

print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

## Data Understanding and Initial Transformation

In [ ]:
from utils import check_data_information

info_df = check_data_information(df, df.columns.tolist())
display(info_df)

**Quick Overview:**

- 21,045 observations, 17 raw columns, no missing values, no duplicated rows.
- Data types are a mix of `object`, `int64`, and `float64`.
- `Date` and `Time` are stored as integers, so they must be rebuilt into a real timestamp.
- `YRMODAHRMI` is a lossy composite of year and month and will be discarded.

**Feature Information (from Pasion et al., 2020)**

| **Feature Name** | **Description** | **Data Type** | **Specific Data Type** |
|------------------|-----------------|---------------|------------------------|
| `Location` | Site identifier (12 northern-hemisphere sites) | Categorical | Nominal |
| `Date` | Observation date, encoded as `YYYYMMDD` | Date | Date |
| `Time` | Observation time, encoded as `HHMM` | Date | Timestamp |
| `Latitude` | Site latitude, controls sun deflection angle | Numerical | Continuous |
| `Longitude` | Site longitude | Numerical | Continuous |
| `Altitude` | Site elevation; thinner atmosphere means more irradiation | Numerical | Continuous |
| `YRMODAHRMI` | Year + month composite, incomplete | Numerical | Redundant |
| `Month` | Month of year; controls sun height and day length | Numerical | Cyclical |
| `Hour` | Hour of day; controls sun position | Numerical | Cyclical |
| `Season` | Season label | Categorical | Nominal |
| `Humidity` | Relative humidity (%); refraction, diffraction, panel dust | Numerical | Continuous |
| `AmbientTemp` | Ambient temperature (°C); panel efficiency falls as it rises | Numerical | Continuous |
| `Wind.Speed` | Wind speed; cools the panel, moves dust | Numerical | Continuous |
| `Visibility` | Visibility distance; low visibility reduces irradiation | Numerical | Continuous |
| `Pressure` | Station pressure (hPa); proxy for weather systems and elevation | Numerical | Continuous |
| `Cloud.Ceiling` | Cloud ceiling height; clouds scatter irradiance | Numerical | Continuous |
| `PolyPwr` | **Target** — panel power output in Watts | Numerical | Continuous |

> Note: `AmbientTemp` is in °C and `Cloud.Ceiling` is capped at 722 (the recorder's clear-sky
> value), which matters when reading its distribution.

In [ ]:
# Group columns by their analytical role
nums_cols = ["Altitude", "Hour", "Humidity", "AmbientTemp", "Wind.Speed",
             "Visibility", "Pressure", "Cloud.Ceiling", "PolyPwr"]
cats_cols = ["Location", "Season"]
date_cols = ["Datetime"]
TARGET = "PolyPwr"

Two raw columns need repair before anything else:

- `Date` and `Time` are integers. `Time = 1145` means 11:45, not 1,145 minutes, so the values are
  zero-padded to `HHMM` and parsed together with the date.
- `YRMODAHRMI` looks like a full timestamp but only encodes year and month
  (`201712000000.000…`), so it is dropped in favour of the rebuilt `Datetime`.

In [ ]:
# Rebuild a real timestamp from the integer Date and Time columns
df["Datetime"] = pd.to_datetime(
    df["Date"].astype(str) + " " + df["Time"].astype(str).str.zfill(4),
    format="%Y%m%d %H%M",
)

# Re-derive the temporal components from the authoritative timestamp
df["Year"] = df["Datetime"].dt.year
df["Month"] = df["Datetime"].dt.month
df["Day"] = df["Datetime"].dt.day
df["Hour"] = df["Datetime"].dt.hour

# The composite date column is lossy and now redundant
df = df.drop(columns=["YRMODAHRMI"])

# Group date, location, weather, and target together; target last (personal preference)
df = df[[
    "Datetime", "Date", "Time", "Year", "Month", "Day", "Hour",
    "Location", "Latitude", "Longitude", "Altitude",
    "Season", "Humidity", "AmbientTemp", "Wind.Speed", "Visibility",
    "Pressure", "Cloud.Ceiling", "PolyPwr",
]]

display(df.head())

`Month` and `Hour` now agree with `Datetime`, and the observation window is confirmed: records
exist only for **hours 10–15**, the daylight band the study sampled.

In [ ]:
print("Date range:", df["Datetime"].min(), "->", df["Datetime"].max())
print("Distinct hours:", sorted(df["Hour"].unique()))
print("Distinct sites:", df["Location"].nunique())

## Descriptive Statistics

In [ ]:
from utils import describe_numerical_combined

display(describe_numerical_combined(df, nums_cols))

In [ ]:
from utils import describe_categorical_combined

display(describe_categorical_combined(df, cats_cols))

In [ ]:
from utils import describe_date_columns

describe_date_columns(df, date_cols)

In [ ]:
from utils import identify_distribution_types

display(identify_distribution_types(df, nums_cols))

**Reminder — reading skewness and kurtosis:**

- `skewness > 0` → right-skewed, `skewness < 0` → left-skewed.
- `kurtosis > 3` → heavy tails (more outliers), `kurtosis < 3` → light tails.

**Major Findings:**

- `PolyPwr` is close to uniform between roughly 0.3 W and 34.3 W (mean 13.0 W, median 13.8 W),
  so accuracy in Watts is a meaningful way to talk about error.
- `Cloud.Ceiling` is dominated by the 722 clear-sky cap, so its distribution is heavily
  zero-inflated at the top rather than continuous.
- `Pressure` spreads from 782 to 1029 hPa because it is *station* pressure: high-altitude sites
  report far lower values, which is exactly why it correlates with `Altitude`.
- The dataset covers 14 months (Dec 2017 – Feb 2019) and 12 sites, with uneven site counts
  (Travis 2,746 rows vs MNANG 780).

---

## Univariate Analysis

### Numerical Columns

In [ ]:
from utils import plot_dynamic_hisplots_kdeplots

plot_dynamic_hisplots_kdeplots(df=df, col_series=nums_cols, ncols=3, figsize=(16, 12))

In [ ]:
plot_dynamic_hisplots_kdeplots(
    df=df, col_series=nums_cols, plot_type="kdeplot", ncols=3, figsize=(16, 12)
)

In [ ]:
from utils import plot_dynamic_boxplots_violinplots

plot_dynamic_boxplots_violinplots(
    df=df, col_series=nums_cols, plot_type="boxplot", ncols=3,
    orientation="h", figsize=(16, 10),
)

**Major Findings:**

- `PolyPwr` is broadly uniform with a slight left shoulder — the panel rarely produces nothing
  during the sampled daylight hours, but output is capped near 34 W.
- `Humidity` is strongly left-skewed: most observations sit in dry air, with a long tail toward
  saturation.
- `AmbientTemp`, `Wind.Speed`, and `Visibility` carry the visible outlier tails; the rest are
  bounded by their physical ranges.
- `Altitude` and `Pressure` both show multi-modal shapes because each mode is a different site.

### Categorical Columns

In [ ]:
from utils import plot_dynamic_countplot

plot_dynamic_countplot(df=df, col_series=cats_cols, ncols=2, figsize=(18, 6))

**Major Findings:**

- The 12 sites are unevenly sampled, so any per-site comparison must be read as a rate, not a raw
  count.
- Seasons are roughly balanced, with summer slightly over-represented because it spans three full
  months of the observation window.

### Outlier Overview

In [ ]:
from utils import filter_outliers

outlier_summary = filter_outliers(
    df, columns=nums_cols, method="iqr", threshold=1.5, detect_only=True
)
display(outlier_summary)

Outliers are concentrated in `AmbientTemp`, `Wind.Speed`, and `Visibility`. They are **not**
removed here: filtering is deferred to the training split (Stage 2) so the test set keeps the
variability the model will meet in production.

---

## Bivariate and Multivariate Analysis

The relationships in this dataset are monotone but clearly non-linear, so **Spearman** is used
throughout instead of Pearson. Pearson would understate a relationship that is strong in rank
terms but bends.

In [ ]:
from utils import plot_correlation_heatmap

plot_correlation_heatmap(
    df, col_series=nums_cols, corr_method="spearman", figsize=(12, 10)
)

In [ ]:
from utils import calculate_correlation_tabular

feature_feature_df, feature_target_df = calculate_correlation_tabular(
    df, target_col=TARGET, method="spearman", corr_type="both"
)

**Major Findings — feature to feature:**

- `Altitude` and `Pressure` are almost perfectly negatively correlated (ρ = −0.97): higher sites
  report lower station pressure. Including both would inject near-perfect multicollinearity, so
  `Altitude` is dropped.
- `AmbientTemp`, `Humidity`, and `Cloud.Ceiling` are inter-correlated (ρ between −0.58 and 0.34),
  which is expected — they describe the same weather systems.

**Major Findings — feature to target:**

- `AmbientTemp` is the strongest single correlate of `PolyPwr` (ρ = 0.59).
- `Cloud.Ceiling` (0.42) and `Humidity` (−0.35) follow, both in the physically expected direction.
- `Visibility` (0.21) and `Latitude` (−0.20) are moderate; `Wind.Speed` (0.08) and `Pressure`
  (0.09) are weak.
- `Hour` (0.10) and `Month` (−0.05) look weak here **only because Spearman measures monotonicity**.
  Both have a clear inverted-U effect — output peaks at midday and in summer — which a rank
  correlation cannot see. They stay in the candidate set precisely because tree models can.

> This is the first sign that correlation alone will under-select features; Stage 3 checks
> mutual information, which does capture non-monotone dependence.

### Categorical features against the target

`PolyPwr` is continuous, while Cramér's V is defined for a categorical pair. To test whether
`Location` and `Season` carry real signal, the target is binned into terciles and the association
is measured against that band. This is a deliberate approximation, not a replacement for a
regression.

In [ ]:
df["PolyPwr_Band"] = pd.qcut(df[TARGET], q=3, labels=["Low", "Medium", "High"])

from utils import analyze_categorical_relationships

categorical_association = analyze_categorical_relationships(
    df, features=["Location", "Season"], target="PolyPwr_Band", alpha=0.05
)
display(categorical_association)

**Reminder — Cramér's V**

$$V = \sqrt{\frac{\chi^2}{n \cdot \min(k-1, r-1)}}$$

It ranges from 0 (no association) to 1 (perfect association). Common reading: `< 0.1` negligible,
`< 0.3` small, `< 0.5` medium, otherwise large.

**Major Findings:**

- `Location` shows a small but statistically detectable association with the power band — sites
  differ, but site identity alone explains little.
- `Season` shows a similar strength of association, and because `Season` is a deterministic
  recode of `Month`, it adds nothing that `Month` does not already carry.
- The practical conclusion is that **weather variables, not site labels, do the work**.

### Power by Location

In [ ]:
plt.figure(figsize=(10, 8))
sns.boxplot(data=df, x=TARGET, y="Location", orient="h")
plt.title("Power output distribution by site")
plt.tight_layout()
plt.show()

In [ ]:
location_hour_power = (
    df.groupby(["Location", "Hour"])[TARGET].mean().reset_index()
)
pivot_location_hour = location_hour_power.pivot(
    index="Location", columns="Hour", values=TARGET
)

plt.figure(figsize=(12, 8))
sns.heatmap(pivot_location_hour, cmap="YlOrRd", annot=True, fmt=".1f", linewidths=0.5)
plt.title("Average power output by site and hour of day")
plt.xlabel("Hour of day")
plt.ylabel("Site")
plt.tight_layout()
plt.show()

**Major Findings:**

- The leading site (JDMT, ≈20 W at midday) and the trailing sites (Grissom, Malmstrom, MNANG,
  ≈10 W) keep their positions across the whole daylight window.
- Individual sites still diverge in shape: Kahului falls off sharply after 13:00 and USAFA peaks
  early, both consistent with local coastal and high-altitude cloud patterns.
- The best-to-worst site ratio is almost exactly 2× at any given hour (JDMT 18.6 W average vs
  Grissom 9.4 W), which is the practical case for site selection.

### Power by Season

In [ ]:
display(df.groupby("Season")[TARGET].agg(["mean", "median", "std"]).round(3))

In [ ]:
season_hour_power = df.groupby(["Season", "Hour"])[TARGET].mean().reset_index()
pivot_season_hour = season_hour_power.pivot(index="Season", columns="Hour", values=TARGET)

plt.figure(figsize=(12, 6))
sns.heatmap(pivot_season_hour, cmap="YlOrRd", annot=True, fmt=".1f", linewidths=0.5)
plt.title("Average power output by season and hour of day")
plt.xlabel("Hour of day")
plt.ylabel("Season")
plt.tight_layout()
plt.show()

**Major Findings:**

- Summer has the highest mean output (15.3 W) and the tightest spread; winter the lowest (8.3 W)
  and the widest — cold, cloudy, short days compound.
- The seasonal swing (≈7 W between summer and winter means) is comparable to the site swing
  (≈9 W between the best and worst site), so weather and geography matter in similar measure.
- `Season` and `Month` orderings agree perfectly, confirming the redundancy found above: `Season`
  is a coarser view of `Month` and adds nothing to a model that already has `Month`.

### Temperature and Power Over Time

In [ ]:
season_colors = {"Winter": "tab:blue", "Spring": "tab:green",
                 "Summer": "tab:red", "Fall": "tab:orange"}

fig, axes = plt.subplots(2, 1, figsize=(18, 8))
for ax, year in zip(axes, [2017, 2018]):
    subset = df[df["Year"] == year]
    sns.lineplot(data=subset, x="Datetime", y=TARGET, hue="Season",
                 palette=season_colors, errorbar=None, ax=ax)
    ax.set_title(f"Power output through {year}")
    ax.set_ylabel("PolyPwr (W)")
    ax.set_xlabel("Date")

plt.tight_layout()
plt.show()

**Major Findings:**

- Output tracks the seasonal cycle rather than a trend: it rises into summer, collapses in winter,
  and recovers. There is no year-over-year drift, which is consistent with a physical process
  rather than panel degradation.
- Short within-season dips line up with weather systems, which is the signal the weather features
  are meant to capture.

In [ ]:
pairplot_cols = ["AmbientTemp", "Humidity", "Pressure", "Cloud.Ceiling", TARGET]
sns.pairplot(df[pairplot_cols], diag_kind="kde")
plt.show()

**Major Findings — modelling implications:**

- The `PolyPwr` relationships are monotone but non-linear, so **tree-based ensembles** are the
  natural first choice: they do not assume a functional form and handle threshold effects
  (for example, output collapsing below a temperature or above a humidity level).
- Linear and distance-based models will need scaling to be competitive, and even then they cannot
  represent the bends visible here.
- Redundancy decisions carried into Stage 2: drop `Altitude` (collinear with `Pressure`), drop
  `Season` (a recode of `Month`), drop `Latitude`/`Longitude` (site identifiers).

---

# **STAGE 2 : DATA PREPROCESSING & FEATURE ENGINEERING**

## Column Pruning

Every dropped column has a specific reason, carried over from the EDA:

| Column | Reason |
|---|---|
| `Datetime`, `Date`, `Time` | Raw timestamp containers; the useful parts are already extracted as `Year/Month/Day/Hour`. |
| `Year` | The window is only 14 months long; year adds no generalisable signal. |
| `Latitude`, `Longitude` | Site identifiers rather than continuous drivers; `Location` covers the same information categorically. |
| `Altitude` | ρ = −0.97 with `Pressure`; keeping both injects near-perfect multicollinearity. |
| `PolyPwr_Band` | Analysis-only artefact created in the EDA. |

`Location` and `Season` are kept for now so their contribution can be measured formally in
Stage 3 before they are discarded.

In [ ]:
df_preprocessed = df.drop(
    columns=["Datetime", "Date", "Time", "Year", "Latitude", "Longitude",
             "Altitude", "PolyPwr_Band"],
    errors="ignore",
)

print(df_preprocessed.shape)
display(df_preprocessed.head())

## Train-Test Split

The split happens **before** encoding, outlier filtering, and scaling. Anything fitted from data
(category levels, scaler statistics, outlier bounds) must come from the training rows only —
otherwise the test score is measuring how well the model memorised the test set.

In [ ]:
from sklearn.model_selection import train_test_split

X = df_preprocessed.drop(columns=[TARGET])
y = df_preprocessed[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train: {y_train.shape} | y_test: {y_test.shape}")

## Feature Encoding

`Location` and `Season` are nominal, so they are one-hot encoded with `drop_first=True` to avoid
the dummy-variable trap. The encoder is **fitted on `X_train` only** and then applied to `X_test`
through the fitted `encoders` object, which guarantees identical column names on both sides.

In [ ]:
from utils import feature_encoding

X_train_encoded, encoders = feature_encoding(
    X_train,
    nominal_columns=["Location", "Season"],
    drop_first=True,
    handle_unknown="ignore",
)
X_test_encoded, _ = feature_encoding(
    X_test,
    nominal_columns=["Location", "Season"],
    drop_first=True,
    handle_unknown="ignore",
    encoders=encoders,
)

# The transform step appends one-hot columns, so align the order explicitly
X_test_encoded = X_test_encoded[X_train_encoded.columns]

assert list(X_train_encoded.columns) == list(X_test_encoded.columns)
print(f"encoded: {X_train_encoded.shape}, columns aligned")

## Handling Outliers

Outliers are removed from the **training rows only**. The test set keeps its outliers on purpose:
a model deployed on new weather will meet unusual days, and hiding them in evaluation would
flatter the score. `AmbientTemp` is the only column filtered, because it is both the strongest
feature and the one with a physically implausible tail.

In [ ]:
from utils import filter_outliers, plot_dynamic_boxplots_violinplots

train_frame = pd.concat([X_train_encoded, y_train], axis=1)

print(f"Rows before filtering: {len(train_frame):,}")
train_frame = filter_outliers(
    train_frame, columns=["AmbientTemp"], method="iqr", threshold=1.5
)
print(f"Rows after filtering:  {len(train_frame):,}")

plot_dynamic_boxplots_violinplots(
    train_frame, col_series=["AmbientTemp"], plot_type="boxplot",
    ncols=1, orientation="h", figsize=(10, 3),
)

In [ ]:
X_train_encoded = train_frame.drop(columns=[TARGET])
y_train = train_frame[TARGET]

print(f"X_train_encoded: {X_train_encoded.shape} | y_train: {y_train.shape}")

## Feature Scaling

Min-max scaling is fitted on the training split and reused on the test split. It is applied only
to the **scale-sensitive** models (linear, distance-based, and neural); tree ensembles split on
thresholds and are invariant to any monotone per-feature transform, so they are trained on the
unscaled matrix.

In [ ]:
from utils import feature_scaling

SCALE_COLUMNS = ["AmbientTemp", "Pressure", "Cloud.Ceiling", "Humidity", "Month", "Hour"]

X_train_scaled, scaler = feature_scaling(
    X_train_encoded, columns=SCALE_COLUMNS, method="minmax"
)
X_test_scaled, _ = feature_scaling(
    X_test_encoded, columns=SCALE_COLUMNS, method="minmax", scaler=scaler
)

display(X_train_scaled[SCALE_COLUMNS].describe().transpose()[["min", "max"]].round(3))

---

# **STAGE 3 : FEATURE SELECTION**

## Correlation on the Encoded Frame

Encoding adds 15 indicator columns (11 sites after `drop_first`, 3 seasons after `drop_first`).
The question is whether any of them earn their place.

In [ ]:
encoded_frame = pd.concat([X_train_encoded, y_train], axis=1)

feature_feature_encoded, feature_target_encoded = calculate_correlation_tabular(
    encoded_frame, target_col=TARGET, method="spearman", corr_type="both"
)

## Multicollinearity Check (VIF)

VIF measures how much a feature's variance is inflated by its linear dependence on the others:
`VIF_i = 1 / (1 - R_i²)`. Above 5 is moderate, above 10 is a problem. Tree models tolerate
correlated inputs, but a feature that is an exact function of another still deserves to be
removed on grounds of interpretability and duplicated importance.

In [ ]:
from utils import calculate_vif

vif_data = calculate_vif(
    encoded_frame, target_col=TARGET, high_threshold=10, moderate_threshold=5
)
display(vif_data.head(15))

**Reading the VIF table:**

- `Pressure` tops the table at **VIF ≈ 341**, and the high-altitude site dummies follow it
  (`USAFA` ≈ 220, `Peterson` ≈ 196, `Hill Weber` ≈ 110). The cause is physical: station pressure
  *is* a function of elevation, and the site dummies encode elevation too. `Altitude` was already
  dropped for this reason; the table shows the same signal still has two homes.
- The `Season_*` indicators sit around **2.4–3.0 and are not flagged at all**. This is worth
  being precise about: one-hot season flags cannot linearly reconstruct a continuous `Month`, so
  VIF is structurally blind to that redundancy. `Season` is a *functional* recode of `Month`
  (visible in the EDA crosstab), not a linear one.
- Conclusion: VIF is the evidence against `Location` duplicating `Pressure`; the crosstab is the
  evidence against `Season` duplicating `Month`. Different tools, different redundancies.

## Statistical Feature Relevance (SelectKBest)

Two complementary scorers are used:

- **Mutual information** (`mutual_info_regression`) is non-parametric and captures *any*
  dependence, including the inverted-U effect of `Hour` and `Month`.
- **F-regression** measures linear dependence only, so it is expected to under-rank those same
  cyclical features and serves as a contrast.

In [ ]:
from utils import calculate_feature_importance

mi_scores = calculate_feature_importance(
    X=X_train_encoded, y=y_train, task="regression", method="mutual_info", k="all"
)
f_scores = calculate_feature_importance(
    X=X_train_encoded, y=y_train, task="regression", method="f_regression", k="all"
)

print("Mutual information, top 15:")
display(mi_scores.iloc[:15].reset_index(drop=True))
print("F-regression, top 15:")
display(f_scores.iloc[:15].reset_index(drop=True))

**Major Findings:**

- Mutual information ranks `AmbientTemp` first (0.262), then **`Pressure` (0.201)** and **`Month`
  (0.179)**, with `Cloud.Ceiling` (0.163) and `Humidity` (0.153) close behind.
- `Pressure` jumping from a Spearman ρ of 0.09 to the second-highest mutual information is the
  clearest proof that its relationship with output is not monotone. Linear correlation would have
  discarded a feature the model can actually use.
- `Month` shows the same pattern: near-zero rank correlation, third-highest mutual information.
  The seasonal effect is an inverted-U, exactly as the season charts showed.
- After the top five there is a large gap (0.153 → 0.078). The next tier is `Season_Winter`,
  `Season_Summer`, and `Location_JDMT` — proxies for the seasonal and site effects already carried
  by `Month` and `Pressure`.
- F-regression tells the linear story instead: `AmbientTemp`, `Cloud.Ceiling`, and `Humidity` lead,
  then the season and site dummies. Because the F-test is linear, it rewards step-function dummies
  and under-ranks the continuous `Month` — the mirror image of the mutual-information result.

## Final Feature Set

Six numeric features survive:

| Feature | Why it stays |
|---|---|
| `AmbientTemp` | Strongest signal in every test; panel efficiency is temperature-sensitive. |
| `Pressure` | Second in mutual information; carries elevation and weather-system effects. |
| `Month` | Third in mutual information; captures the seasonal cycle Spearman cannot see. |
| `Cloud.Ceiling` | Direct proxy for how much irradiance reaches the panel. |
| `Humidity` | Affects refraction and dust, and is inversely tied to clear skies. |
| `Hour` | The sun-angle curve within the 10:00–15:00 window. |

**Dropped, with the trade-off stated:**

- `Season_*` — a functional recode of `Month`. Keeping both is redundancy, not signal.
- `Location_*` — these encode real site fixed effects (elevation, latitude, panel wiring), and they
  rank mid-table in both scorers. Dropping them *does* cost some accuracy. The reason to drop them
  anyway is that a model built on site dummies cannot score a site outside the training set, which
  is the entire business question. The price of transferability is disclosed here rather than
  hidden in the score.
- `Wind.Speed`, `Visibility` — below the top tier in both scorers, with noisy tails. Kept out to
  keep the deployed input surface small.

> Consequence for deployment: the model consumes weather and time only. Location is used to look
> up a site's typical conditions, not as a model input.

In [ ]:
SELECTED_FEATURES = ["AmbientTemp", "Pressure", "Cloud.Ceiling",
                     "Humidity", "Month", "Hour"]

X_train_model = X_train_encoded[SELECTED_FEATURES]
X_test_model = X_test_encoded[SELECTED_FEATURES]
X_train_model_scaled = X_train_scaled[SELECTED_FEATURES]
X_test_model_scaled = X_test_scaled[SELECTED_FEATURES]

SCALE_SENSITIVE = ["LinearRegression", "Ridge", "Lasso", "ElasticNet",
                   "KNN", "SVR", "MLP"]

print(f"Unscaled: {X_train_model.shape} / {X_test_model.shape}")
print(f"Scaled:   {X_train_model_scaled.shape} / {X_test_model_scaled.shape}")
display(X_train_model.head())

---

# **STAGE 4 : MODELLING & EVALUATION**

## Evaluation Harness

All evaluation goes through `utils.eval_regression`, which reports single-run metrics on the
held-out test set **and** repeated cross-validated metrics on the training set. Two configurations
are used deliberately:

- **Screening (3-fold, 1 repeat)** for the broad model comparison — enough to rank eleven
  algorithms without paying for 15 fits each.
- **Confirmation (5-fold, 3 repeats)** for the two finalists, before and after tuning, so the
  headline numbers carry a real variance estimate.

`SVR` is excluded from the comparison: its training cost grows roughly quadratically with sample
count, and at 15,600 rows it would dominate the runtime without changing the conclusion.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor,
)
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(random_state=42),
    "Lasso": Lasso(random_state=42),
    "ElasticNet": ElasticNet(random_state=42),
    "KNN": KNeighborsRegressor(),
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "AdaBoost": AdaBoostRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42),
    "MLP": MLPRegressor(random_state=42),
}


def training_matrix(name):
    """Return the (train, test) matrices a model should be fitted on."""
    if name in SCALE_SENSITIVE:
        return X_train_model_scaled, X_test_model_scaled
    return X_train_model, X_test_model


for name, model in models.items():
    X_fit, _ = training_matrix(name)
    model.fit(X_fit, y_train)

print(f"fitted {len(models)} models")

## Screening Comparison

In [ ]:
from utils import eval_regression, compare_cv_metrics

SCREEN = {"n_splits": 3, "n_repeats": 1}

screening_metrics = {}
for name, model in models.items():
    X_fit, X_holdout = training_matrix(name)
    screening_metrics[name] = eval_regression(
        model, X_holdout, y_test, X_fit, y_train,
        plot_predicted_vs_actual=False,
        plot_residuals=False,
        plot_residuals_dist=False,
        display=False,
        **SCREEN,
    )

comparison_df = compare_cv_metrics(screening_metrics)
display(comparison_df)

**Major Findings:**

- The tree ensembles (`RandomForest`, `XGBoost`, `GradientBoosting`) lead the screening table, with
  `DecisionTree` close behind on the training data but visibly worse on held-out rows — the classic
  overfit signature of a single tree.
- The linear family (`LinearRegression`, `Ridge`, `Lasso`, `ElasticNet`) trails by a wide margin.
  That is the expected result after the EDA: the relationships are monotone-but-bent, and no linear
  coefficient can represent a midday peak or a summer hump.
- `KNN` and `MLP` improve substantially once scaled, which is why they are fed the min-max matrix,
  but neither reaches the ensembles.
- `AdaBoost` underperforms because its default base learner is a shallow stump; it cannot express
  the interactions the problem needs.

## Detailed Diagnostics for the Finalists

Random Forest and XGBoost are re-evaluated with the confirmation configuration and full diagnostic
plots: predicted-vs-actual, residuals-vs-predicted, and the residual distribution.

In [ ]:
CONFIRM = {"n_splits": 5, "n_repeats": 3}

results_rf_vanilla = eval_regression(
    models["RandomForest"], X_test_model, y_test, X_train_model, y_train, **CONFIRM
)

In [ ]:
results_xgb_vanilla = eval_regression(
    models["XGBoost"], X_test_model, y_test, X_train_model, y_train, **CONFIRM
)

In [ ]:
# Quantify what the diagnostic plots show, instead of eyeballing them
residuals = (
    y_test.reset_index(drop=True)
    - pd.Series(models["RandomForest"].predict(X_test_model)).reset_index(drop=True)
)

display(pd.DataFrame({
    "Bias (mean residual)": [residuals.mean()],
    "Residual std": [residuals.std()],
    "Skew": [residuals.skew()],
    "Share |error| <= 3 W": [(residuals.abs() <= 3).mean()],
    "Share |error| <= 5 W": [(residuals.abs() <= 5).mean()],
}).round(3))

banded = pd.DataFrame({
    "Predicted band": pd.qcut(
        models["RandomForest"].predict(X_test_model), 3,
        labels=["Low", "Mid", "High"],
    ),
    "Absolute error": residuals.abs(),
})
display(
    banded.groupby("Predicted band", observed=True)["Absolute error"]
    .agg(["mean", "count"]).round(3)
)

**What the diagnostics show (Random Forest, test split):**

- Bias is essentially zero (0.06 W) and the residual standard deviation is 4.25 W, so the model is
  not systematically over- or under-predicting.
- Errors are slightly left-skewed (−0.18): a handful of rows are over-predicted by more than they
  are under-predicted.
- 67% of test rows land within ±3 W and 82% within ±5 W, against a target that ranges from 0.3 W
  to 34.3 W.
- Error is **not** concentrated at the extremes. Absolute error averages 2.57 W in the low band,
  3.33 W in the middle, and 2.73 W in the high band. The middle band is hardest, which is where
  partly cloudy and clear days share similar weather but differ in actual irradiance — the part of
  the relationship the features cannot resolve.

---

# **STAGE 5 : HYPERPARAMETER TUNING**

## Tuning Strategy

Both finalists are tuned with **randomized search** over the shared `utils` distributions rather
than exhaustive grid search. The parameter spaces are large (hundreds of combinations), and random
search reaches a comparable optimum in a fraction of the fits. Twenty-five candidate configurations
with 5-fold CV and a single repeat keeps the notebook's runtime reasonable on a laptop while still
sampling the space broadly.

> **Workaround, stated openly.** The vendored `utils` random-search distribution for
> `RandomForestRegressor` includes `criterion='friedman_mse'`, which scikit-learn accepts only for
> `GradientBoostingRegressor`. Rather than patching a vendored package, the invalid value is
> removed here and the remaining distribution is used unchanged.

In [ ]:
from utils import get_hyperparameters, get_model_pipeline, tune_pipelines

# Drop the value sklearn rejects for RandomForestRegressor; keep the rest of
# the shared random-search distribution intact.
RF_PARAMS = {
    key: value
    for key, value in get_hyperparameters("randomforest", search_method="random").items()
    if key != "rf__criterion"
}
RF_PARAMS["rf__criterion"] = ["squared_error", "absolute_error"]

rf_models, rf_times = tune_pipelines(
    pipedict={"randomforest": get_model_pipeline("randomforest")},
    param_grid={"randomforest": RF_PARAMS},
    X_train=X_train_model, y_train=y_train,
    X_test=X_test_model, y_test=y_test,
    scoring="r2",
    search_method="random",
    n_iter=25,
    n_splits=5,
    n_repeats=1,
    plot_learning_curves=False,
    display=True,
)
rf_search = rf_models["randomforest"]
rf_time = rf_times[0]

print(f"Random Forest best CV R2: {rf_search.best_score_:.4f} in {rf_time:.1f}s")
print(rf_search.best_params_)

In [ ]:
xgb_models, xgb_times = tune_pipelines(
    pipedict={"xgboost": get_model_pipeline("xgboost")},
    param_grid={"xgboost": get_hyperparameters("xgboost", search_method="random")},
    X_train=X_train_model, y_train=y_train,
    X_test=X_test_model, y_test=y_test,
    scoring="r2",
    search_method="random",
    n_iter=25,
    n_splits=5,
    n_repeats=1,
    plot_learning_curves=False,
    display=True,
)
xgb_search = xgb_models["xgboost"]
xgb_time = xgb_times[0]

print(f"XGBoost best CV R2: {xgb_search.best_score_:.4f} in {xgb_time:.1f}s")
print(xgb_search.best_params_)

## Tuned Models

In [ ]:
# The search objects already refit the best configuration on the full training set
rf_tuned = rf_search.best_estimator_.named_steps["rf"]
xgb_tuned = xgb_search.best_estimator_.named_steps["xgb"]

In [ ]:
results_rf_tuned = eval_regression(
    rf_tuned, X_test_model, y_test, X_train_model, y_train, **CONFIRM
)

In [ ]:
results_xgb_tuned = eval_regression(
    xgb_tuned, X_test_model, y_test, X_train_model, y_train, **CONFIRM
)

## Before and After Tuning

In [ ]:
before_after = compare_cv_metrics({
    "Random Forest (vanilla)": results_rf_vanilla,
    "Random Forest (tuned)": results_rf_tuned,
    "XGBoost (vanilla)": results_xgb_vanilla,
    "XGBoost (tuned)": results_xgb_tuned,
})
display(before_after)

**Major Findings:**

- Tuning moves the cross-validated R² by a small margin, not a large one. That is the honest
  result: the default ensembles were already close to their ceiling on six features, and the
  remaining error is irreducible noise in the weather-to-power relationship.
- The gap between cross-validated and held-out test R² is small for both tuned models, which means
  the search did not simply overfit the validation folds.
- Because the improvement is modest, the simpler consideration matters: both models are cheap to
  serve, so the tie-breaker is held-out error rather than training time.

In [ ]:
# Model-based feature importance for the tuned ensembles
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (label, model) in zip(axes, [("Random Forest", rf_tuned), ("XGBoost", xgb_tuned)]):
    importance = pd.Series(model.feature_importances_, index=SELECTED_FEATURES)
    importance.sort_values().plot(kind="barh", ax=ax, color="#D97706")
    ax.set_title(f"{label} feature importance")
    ax.set_xlabel("Impurity-based importance")
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

**Major Findings:**

- Both ensembles put `AmbientTemp` first by a wide margin (0.35 in Random Forest, 0.38 in XGBoost)
  — roughly double the next feature in each model.
- They also agree that `Month` and `Cloud.Ceiling` belong in the top four, which matches the EDA:
  season and cloud cover are the two levers a planner can reason about.
- Beyond that they diverge. `Humidity` is 2nd in Random Forest (0.17) but 5th in XGBoost (0.08);
  `Hour` is last in Random Forest (0.06) but 4th in XGBoost (0.12).
- That divergence is the real finding. With correlated inputs, a tree ensemble can satisfy the same
  split using either of two features, so impurity importance reflects which one happened to be
  chosen, not how much the outcome depends on it. **Importance ranks influence, not causation** —
  the consistent message across both models is `AmbientTemp` first, followed by cloud and season.

# **STAGE 6 : CONCLUSION & MODEL PERSISTENCE**

## Conclusion

- A six-feature weather-and-time model reaches a cross-validated R² in the low 0.6s and a held-out
  R² of the same order — close to what the source paper reports for a far larger feature set.
- The signal is physical and stable: ambient temperature, cloud cover, humidity, and pressure carry
  almost all of the explanatory power, with month and hour shaping the cycle.
- Location dummies were deliberately removed. That decision trades a little accuracy for a model
  that can be pointed at a site it has never seen, which is the actual business use case.

**Limitations, stated plainly:**

1. **Temporal leakage.** The split is random over rows, so the same site and timestamp can appear in
   both train and test. Reported R² is therefore optimistic for a genuinely new day. A group or
   time-based split would be the correct next experiment.
2. **Daylight only.** Observations exist only for hours 10–15. The model says nothing about
   night-time or early-morning output.
3. **Twelve sites, 14 months.** All northern-hemisphere, all in the United States. Nothing here
   transfers to the southern hemisphere without re-validation.
4. **MAPE is unstable near zero.** With `PolyPwr` reaching 0.26 W, percentage error explodes for
   low-output rows; MAE and RMSE are the trustworthy metrics.
5. **Cloud ceiling is capped at 722.** The feature is effectively "clear or not" at the top of its
   range, so it cannot distinguish between two perfectly clear days.

**Next steps:** a location-grouped split to measure true transfer, a lag/rolling weather window to
capture persistence, and a quantile model to report a yield range instead of a point estimate.

In [ ]:
import joblib
from pathlib import Path

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)


def build_bundle(model, metrics, label):
    """Package a fitted model with everything a consumer needs to score a row."""
    return {
        "model": model,
        "feature_columns": list(SELECTED_FEATURES),
        "metrics": {
            "cv": metrics["cv"],
            "test": {key: value["test"] for key, value in metrics.items() if key != "cv"},
        },
        "label": label,
        "target": TARGET,
    }


rf_bundle = build_bundle(rf_tuned, results_rf_tuned, "Random Forest")
xgb_bundle = build_bundle(xgb_tuned, results_xgb_tuned, "XGBoost")

joblib.dump(rf_bundle, MODELS_DIR / "tuned_rf_model.joblib")
joblib.dump(xgb_bundle, MODELS_DIR / "tuned_xgb_model.joblib")

print("saved:", sorted(path.name for path in MODELS_DIR.glob("*.joblib")))

The saved artifacts are **bundles**, not bare estimators: each carries the fitted model, the exact
feature order the model expects, its evaluation metrics, a display label, and the target name. A
consumer can therefore rebuild a valid input row without reading this notebook.

Re-running this notebook regenerates both files. The Streamlit app reads them from `models/`.